# Segundo experimento RNN: selección por F1 positiva

Este notebook entrena `rnn_reference_f1_v1` y guarda la corrida `rnn_f1_v1_seed42`, con el código compartido del repositorio. Mantiene los datos, arquitectura, máscaras, W=8, semilla 42 y entrenamiento de la referencia F2 `rnn_reference_v1` (`runs/rnn_v1_seed42`). Solo cambia el criterio de selección: **F2 → F1**. Train sigue siendo 2022–2023 y validación 2024. **2025 permanece reservada.**

La pérdida diferenciable sigue siendo `BCEWithLogitsLoss` sin ponderación. F1 es el criterio para elegir el checkpoint a umbral fijo 0,5, con menor BCE de validación como desempate, y para elegir después el umbral con esos pesos congelados. Se reportan F1 y F2 para poder comparar ambos experimentos.

El control anterior que busca F1 con el checkpoint F2 de la época 10 congelado aísla un cambio de umbral: eligió 0,22 sobre esas mismas probabilidades. Se documenta en `docs/aclaraciones/07_control_f1_sobre_pesos_f2.md` y `docs/aclaraciones/evidencia_control_f1_v1/control.json`. Esta corrida vuelve a entrenar y selecciona época/parada por F1; permite examinar el efecto adicional de la selección de pesos. No se presupone una mejora.

El notebook se distribuye sin ejecuciones ni métricas. Modelo, entrenamiento y evaluación se reutilizan desde módulos y comandos comunes.

## 1. Preparar el paquete localmente

Desde la raíz del repositorio, ejecutar **antes de subir archivos a Colab**:

```bash
python scripts/empaquetar_experimento_rnn.py \
  --config configs/experimento_rnn_f1_v1.json \
  --output runs/rnn_colab_f1_v1.zip
```

Guardar el SHA-256 que muestre el empaquetador para contrastarlo aquí. Ese ZIP lleva código, configuración, dependencias y los artefactos necesarios de train/validación. Tras extraer, se exige `colab_package_manifest.json` versión `rnn_colab_inputs_v1`, `test_arrays_included=false`, conjunto exacto de archivos y SHA-256 correcto de cada archivo. No hay que cargar datos adicionales ni archivos de prueba 2025.

Abrir este notebook en Google Colab. Para usar GPU: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. El dispositivo `auto` utiliza CUDA si está disponible y CPU en caso contrario. Google Drive es opcional.

Subir primero el ZIP al panel **Archivos** de Colab deja el archivo en `/content/rnn_colab_f1_v1.zip`. La celda siguiente lo reutiliza si ya existe; solo llama a `files.upload()` como alternativa cuando falta. Si el archivo tiene otro nombre, ajustar `PACKAGE_PATH`.

Ejecutar las siguientes celdas en orden desde una sesión limpia. Si `/content/f1-rnn-f1` ya existe, iniciar una sesión nueva para evitar mezclar corridas.

In [ ]:
from pathlib import Path
import hashlib
import json
import stat
import subprocess
import sys
import zipfile

from google.colab import files

# Opcional: copiar aquí el SHA-256 calculado localmente.
EXPECTED_PACKAGE_SHA256 = ""
ROOT = Path("/content/f1-rnn-f1")
CONFIG_RELATIVE_PATH = Path("configs/experimento_rnn_f1_v1.json")
# Si el ZIP ya está en el panel Archivos, se lee directamente sin volver a subirlo.
PACKAGE_PATH = Path("/content/rnn_colab_f1_v1.zip")

if ROOT.exists():
    raise FileExistsError("/content/f1-rnn-f1 ya existe. Usa una sesión limpia para esta corrida.")
if PACKAGE_PATH.is_file():
    print("Se reutiliza el ZIP ya subido:", PACKAGE_PATH)
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Sube exactamente un archivo ZIP generado por el empaquetador F1.")
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    if Path(uploaded_name).suffix.lower() != ".zip":
        raise ValueError("El archivo debe ser el ZIP generado para el experimento F1.")
    PACKAGE_PATH.write_bytes(uploaded_bytes)
    del uploaded_bytes, uploaded

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as input_file:
        for chunk in iter(lambda: input_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

package_sha256 = file_sha256(PACKAGE_PATH)
print("SHA-256 del paquete:", package_sha256)
if EXPECTED_PACKAGE_SHA256 and package_sha256 != EXPECTED_PACKAGE_SHA256.strip().lower():
    raise ValueError("El SHA-256 no coincide con el paquete local.")

# Validar todos los nombres antes de extraer: sin rutas que salgan del destino,
# enlaces simbólicos ni destinos repetidos.
with zipfile.ZipFile(PACKAGE_PATH) as archive:
    members = archive.infolist()
    destinations = set()
    for member in members:
        name = member.filename
        if "\\" in name:
            raise ValueError(f"Ruta ZIP no admitida: {name!r}")
        candidate = Path(name)
        if candidate.is_absolute() or ".." in candidate.parts:
            raise ValueError(f"Ruta ZIP fuera del destino: {name!r}")
        if stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError(f"Enlace simbólico no admitido: {name!r}")
        destination = (ROOT / candidate).resolve()
        if not destination.is_relative_to(ROOT.resolve()) or destination == ROOT.resolve():
            raise ValueError(f"Ruta ZIP no admitida: {name!r}")
        if destination in destinations:
            raise ValueError(f"Destino ZIP repetido: {name!r}")
        destinations.add(destination)
    ROOT.mkdir(parents=True)
    archive.extractall(ROOT)

# Comprobar el contrato y todos los hashes del paquete antes de instalar o
# importar su código. El manifiesto no se incluye en su propio mapa de hashes.
package_manifest_path = ROOT / "colab_package_manifest.json"
package_manifest = json.loads(package_manifest_path.read_text(encoding="utf-8"))
if package_manifest.get("artifact_version") != "rnn_colab_inputs_v1":
    raise ValueError("Versión de paquete Colab no admitida.")
if package_manifest.get("test_arrays_included") is not False:
    raise ValueError("El paquete debe declarar test_arrays_included=false.")
if package_manifest.get("experiment_config") != CONFIG_RELATIVE_PATH.as_posix():
    raise ValueError("El paquete no declara la configuración F1 esperada.")
file_hashes = package_manifest.get("files_sha256")
if not isinstance(file_hashes, dict) or not file_hashes:
    raise ValueError("El manifiesto debe enumerar los archivos y sus SHA-256.")
for name, expected_hash in file_hashes.items():
    if not isinstance(name, str) or "\\" in name:
        raise ValueError(f"Ruta de manifiesto no admitida: {name!r}")
    relative = Path(name)
    resolved = (ROOT / relative).resolve()
    if relative.is_absolute() or ".." in relative.parts or not resolved.is_relative_to(ROOT.resolve()):
        raise ValueError(f"Ruta de manifiesto fuera del destino: {name!r}")
    if resolved == ROOT.resolve() or resolved == package_manifest_path.resolve():
        raise ValueError(f"Ruta de manifiesto no admitida: {name!r}")
    if not isinstance(expected_hash, str) or len(expected_hash) != 64:
        raise ValueError(f"SHA-256 no válido para {name!r}")
    if any(character not in "0123456789abcdef" for character in expected_hash):
        raise ValueError(f"SHA-256 no válido para {name!r}")

expected_files = set(file_hashes) | {"colab_package_manifest.json"}
actual_files = {file.relative_to(ROOT).as_posix() for file in ROOT.rglob("*") if file.is_file()}
if actual_files != expected_files:
    raise ValueError(
        "Archivos distintos del manifiesto: "
        f"faltantes={sorted(expected_files - actual_files)}, "
        f"adicionales={sorted(actual_files - expected_files)}"
    )
for name, expected_hash in file_hashes.items():
    if file_sha256(ROOT / name) != expected_hash:
        raise ValueError(f"El SHA-256 no coincide para {name!r}")
print("Manifiesto y SHA-256 verificados:", len(file_hashes), "archivos; sin arrays de prueba.")

required = [
    "requirements-experiments.txt",
    CONFIG_RELATIVE_PATH.as_posix(),
    "scripts/entrenar_rnn.py",
    "scripts/evaluar_rnn.py",
    "src/f1_pitstop/rnn_model.py",
    "EDA/dataset/muestras_preprocesadas_v1_3/preprocessor.json",
    "EDA/dataset/muestras_preprocesadas_v1_3/manifest.json",
]
missing = [name for name in required if not (ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Paquete incompleto: {missing}")
print("Paquete extraído en", ROOT)


## 2. Instalar dependencias y revisar la configuración

La instalación respeta los rangos del archivo de requisitos y permite conservar un PyTorch CUDA compatible de Colab. No se fuerza un índice de wheels CPU. La exportación registra las versiones exactas empleadas; el hash del paquete identifica los archivos cargados.

La arquitectura se conserva frente a la referencia F2: usa una sola `RNNCell` tanh de 32 unidades, dos embeddings dinámicos de dimensión 4 y una rama estática mínima de 16 unidades. A los 22 valores numéricos se añaden explícitamente 22 bits de disponibilidad de sus máscaras. Los conteos y fracciones de cobertura de QA no son entradas.

La configuración fija semilla 42, Adam con learning rate 0,001, BCE sin ponderación inicial, batch 256, máximo 30 épocas, paciencia 6 y recorte de gradiente a 1. El único factor modificado es la selección por F1; BCE, arquitectura y parámetros de entrenamiento permanecen iguales a F2.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements-experiments.txt")],
    check=True,
)


In [ ]:
from importlib import metadata

sys.path.insert(0, str(ROOT / "src"))
import torch
from f1_pitstop.rnn_model import SmallPitStopRNN

CONFIG_PATH = ROOT / CONFIG_RELATIVE_PATH
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
if config.get("experiment_id") != "rnn_reference_f1_v1":
    raise ValueError("La configuración no corresponde al segundo experimento F1.")
if config["selection"]["primary_metric"] != "positive_class_f1":
    raise ValueError("La métrica principal debe ser positive_class_f1.")
print(json.dumps(config, ensure_ascii=False, indent=2))
print("Código compartido:", SmallPitStopRNN.__module__ + "." + SmallPitStopRNN.__name__)
print("Python:", sys.version)
for dependency in ("torch", "numpy", "pandas"):
    print(dependency + ":", metadata.version(dependency))
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Se entrenará en CPU; esta ejecución puede tardar más.")


## 3. Entrenar mediante el comando compartido

El loader comprueba hashes, alineación, esquema y población supervisada antes de entrenar. Solo se consume train y validación. El preprocesador ya fue ajustado con 46.162 muestras elegibles de train; aquí se reutiliza su estado.

El checkpoint se elige con F1 positiva de validación a umbral **fijo 0,5**, con menor pérdida de validación como desempate. La paciencia observa ese mismo criterio. Una vez congelado el checkpoint, se maximiza F1 al seleccionar el umbral en `0,05, 0,06, …, 0,95`; los empates se resuelven por mayor precision y luego mayor umbral. No se selecciona una época diferente tras buscar el umbral.

La pérdida sigue siendo BCE sin ponderación: F1 no sustituye a la función de pérdida. La selección por F1 puede cambiar el checkpoint elegido y la época de parada respecto de F2.

La salida debe ser nueva. El comando genera tanto la carpeta de artefactos como su ZIP hermano.

In [ ]:
RUN_DIR = ROOT / "runs/rnn_f1_v1_seed42"
RUN_ZIP = RUN_DIR.with_suffix(".zip")
if RUN_DIR.exists() or RUN_ZIP.exists():
    raise FileExistsError("La salida de esta corrida ya existe. Conserva sus artefactos antes de repetir.")

subprocess.run(
    [
        sys.executable,
        "scripts/entrenar_rnn.py",
        "--config", CONFIG_RELATIVE_PATH.as_posix(),
        "--output", str(RUN_DIR.relative_to(ROOT)),
    ],
    cwd=ROOT,
    check=True,
)


## 4. Inspeccionar la evidencia

El historial permite revisar el aprendizaje y el checkpoint elegido. Las métricas incluyen el resultado a umbral 0,5 y a umbral seleccionado sobre las mismas probabilidades. La diferencia representa el efecto de cambiar la regla de alerta con pesos congelados. Comparar F1 y F2 tanto a 0,5 como al umbral elegido; conservar también la referencia F2 y el control F1 calculado sobre su checkpoint congelado. Ese control cambia la decisión, mientras esta corrida permite cambiar además la época seleccionada.

Revisar precision, recall, F1, F2, matriz de confusión y cantidad de muestras elegibles. Estas métricas de entrenamiento se calculan por muestra piloto-vuelta. La cobertura por evento se calcula después, únicamente en el evaluador local, con las fuentes históricas verificadas por hash. El resultado ajustado en validación orienta desarrollo; no sustituye una evaluación final independiente. Una única semilla tampoco permite estimar variabilidad entre corridas.

In [ ]:
import pandas as pd
from IPython.display import display

history_path = RUN_DIR / "history.csv"
metrics_path = RUN_DIR / "metrics.json"
display(pd.read_csv(history_path))
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("Artefactos:")
for artifact in sorted(RUN_DIR.iterdir()):
    if artifact.is_file():
        print(" -", artifact.name)


## 5. Descargar la corrida

El ZIP conserva pesos (`state_dict`), arquitectura, configuración, preprocesador, umbral, entorno exacto, historial, métricas, predicciones de validación por `sample_id` y manifiesto con hashes. Guardarlo antes de cerrar Colab.

La descarga utiliza el ZIP que generó el entrenamiento, sin reconstruir ni modificar sus artefactos.

In [ ]:
if not RUN_ZIP.is_file():
    raise FileNotFoundError(f"El entrenamiento no generó el ZIP esperado: {RUN_ZIP}")
print("Descarga:", RUN_ZIP)
files.download(str(RUN_ZIP))


### Copia opcional en Google Drive

Esta celda está desactivada por defecto. Para conservar una segunda copia en Drive, cambiar `COPY_TO_DRIVE` a `True`; Colab solicitará acceso al montar Drive. No hace falta para descargar el ZIP ni evaluar localmente.

In [ ]:
COPY_TO_DRIVE = False
if COPY_TO_DRIVE:
    import shutil
    from google.colab import drive

    drive.mount("/content/drive")
    drive_directory = Path("/content/drive/MyDrive/F1-PitStop-Predictor")
    drive_directory.mkdir(parents=True, exist_ok=True)
    drive_target = drive_directory / RUN_ZIP.name
    if drive_target.exists():
        raise FileExistsError(f"Ya existe una copia: {drive_target}")
    shutil.copy2(RUN_ZIP, drive_target)
    print("Copia conservada en", drive_target)


## 6. Volver al repositorio y evaluar en CPU

Extraer el ZIP descargado. Desde la raíz del repositorio, usar como `--bundle` la **carpeta que contiene `weights.pt`**, no el ZIP:

```bash
python scripts/evaluar_rnn.py \
  --bundle RUTA_DIRECTORIO_MODELO \
  --data EDA/dataset/muestras_preprocesadas_v1_3 \
  --output eval/rnn_f1_v1_validation \
  --device cpu
```

El evaluador local también necesita las fuentes que ya existen en el repositorio: `events.csv` de la publicación `etiquetado_pit_entry_v3_2022_2025` y `sample_index.csv.gz` de `muestras_causales_v1_3`, en las rutas fijadas en la configuración. Ambas se verifican por hash y **no se incluyen en el paquete Colab**. Se usan para calcular la cobertura por evento de validación, únicamente en esta evaluación local.

El evaluador carga el mismo modelo, pesos, preprocesador y umbral congelados; contrasta probabilidades de validación por `sample_id` con las exportadas por Colab. Usa `parity_atol=1e-5` y `parity_rtol=1e-4`, registradas en configuración y artefactos, sin exigir igualdad bit a bit entre GPU y CPU. Si la comparación falla, se investiga la discrepancia antes de publicar métricas. Scores muy próximos al umbral pueden cambiar de lado dentro de la tolerancia; el evaluador registra las diferencias y cuenta esos cambios de decisión.

Conservar esta corrida, el paquete F2 original y el control de umbral F1 con pesos F2 congelados. La comparación debe mostrar F1, F2, precision y recall, sin atribuir una diferencia de umbral a la arquitectura. Para la siguiente iteración, declarar el factor que se modifica: capacidad, entradas, entrenamiento o decisión. Comparar umbrales mantiene pesos congelados; comparar aprendizaje mantiene el protocolo de selección. **2025 sigue reservada para la evaluación final acordada.**

Contrato completo: `docs/specs/12_experimento_rnn_f1.md`.